
# Attention Visualization on a Contract Sentence

**Day 1 — AI Foundations · Practical 1 of 6 · Companion to the "Transformers 101" deck**

> **Running in Google Colab:** this notebook works fine on the default **CPU runtime** — no
> GPU needed. Just run cells top to bottom; the first code cell installs everything required.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Load a pretrained transformer model and extract its internal **self-attention weights**
2. Visualize **which words a model attends to** when interpreting an ambiguous legal pronoun
3. Compare attention patterns across **multiple heads** to see how different heads specialize
4. Connect what you see here back to the math from the Transformers 101 slides:
   `Attention(Q, K, V) = softmax(QKᵀ / √d_k) · V`

## Why This Matters for a Law Firm

Contracts are full of pronouns and cross-references — *"it," "the party," "such obligation,"
"the foregoing"* — whose correct referent is often the entire legal question. A transformer
resolves this ambiguity purely through **self-attention**: every token computes a relationship
score with every other token in the sentence. This notebook makes that abstract mechanism
visible and concrete using a real indemnification-clause-style sentence.

## Notebook Workflow

```mermaid
flowchart LR
    A["Contract sentence\n(raw text)"] --> B["Tokenizer\n(WordPiece)"]
    B --> C["DistilBERT Encoder\n(6 layers x 12 heads)"]
    C --> D["Attention weight\nmatrices per layer/head"]
    D --> E["Heatmap\nvisualization"]
    D --> F["Which token does\n'it' attend to?"]
    E --> G["Compare heads:\nsyntax vs. reference"]
    F --> G
    G --> H["Takeaways ->\nlink back to Transformers 101 deck"]
```

---



## Section 1 — Setup

We use **DistilBERT** (`distilbert-base-uncased`) rather than DistilGPT2 here because
DistilBERT is a **bidirectional encoder** — every token can attend to every other token in
both directions (left and right). This matches how a human lawyer reads a sentence: you don't
resolve "it" by only looking backward, you use the *whole* sentence. GPT-style decoder models
are **masked** (causal) and can only look backward — great for generation, less intuitive for
a first look at "what resolves this pronoun."

We ask the model to return `output_attentions=True`, which gives us the raw softmax attention
weight matrices for every layer and every head — exactly the `softmax(QKᵀ/√d_k)` term from the
Transformers 101 deck, before it gets multiplied by `V`.


In [ ]:

# Install dependencies.
# Running in Google Colab: this cell installs everything needed -- just run it.
# Running locally with these packages already installed: safe to run anyway (no-op if current).
%pip install -q transformers torch matplotlib

import torch
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility: pin the random seed so any stochastic steps behave consistently
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")



## Section 2 — Load the Model and Tokenizer

`AutoModel.from_pretrained(..., output_attentions=True)` loads DistilBERT and configures it to
return attention weights alongside its normal output. This is the same
`distilbert-base-uncased` checkpoint used broadly across the NLP ecosystem — small enough to
run on a laptop CPU, large enough to show real, non-trivial attention patterns.


In [ ]:

MODEL_NAME = "distilbert-base-uncased"

# The tokenizer converts raw text into the WordPiece subword tokens the model expects
# (see the companion Tokenization notebook for how this subword vocabulary is built)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# output_attentions=True is the key flag: it tells the model to return every layer's
# softmax(QK^T / sqrt(d_k)) attention matrix, not just the final hidden states
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()  # inference mode: disables dropout so results are deterministic

print(f"Loaded {MODEL_NAME}")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Attention heads per layer: {model.config.num_attention_heads}")
print(f"  Hidden size (d_model): {model.config.hidden_size}")



## Section 3 — The Contract Sentence

The sentence below is written in the style of a real indemnification clause (the pattern is
standard boilerplate seen across countless commercial agreements, including examples in the
CUAD — Contract Understanding Atticus Dataset — indemnification category). The pronoun **"it"**
is the target of our investigation: does the model's attention correctly connect "it" back to
**"any claim"**, or does it get confused with "the Lessee" or "the Lessor"?

> *"The Lessee shall indemnify the Lessor against any claim arising therefrom, provided that
> it was not caused by gross negligence."*

This is exactly the kind of sentence a junior associate has to parse carefully — and exactly
the kind of sentence a transformer resolves via self-attention in a single forward pass.


In [ ]:

sentence = (
    "The Lessee shall indemnify the Lessor against any claim arising therefrom, "
    "provided that it was not caused by gross negligence."
)

# Tokenize with return_tensors="pt" so we get PyTorch tensors ready for the model
inputs = tokenizer(sentence, return_tensors="pt")

# Recover the human-readable subword tokens for labeling our plots later
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Token count:", len(tokens))
print(tokens)



## Section 4 — Run the Model and Extract Attention Weights

A single forward pass (`model(**inputs)`) computes everything: embeddings, positional
encoding, and all 6 transformer layers. Because `output_attentions=True`, the output object
also contains `outputs.attentions` — a tuple with one tensor per layer.

Each tensor has shape `(batch_size, num_heads, seq_len, seq_len)`. The value at
`[batch, head, i, j]` is **how much token `i` attends to token `j`** — the softmax output row
for token `i` sums to 1 across all `j`, exactly matching the attention formula from the deck.


In [ ]:

with torch.no_grad():  # no need to track gradients — we're only doing inference
    outputs = model(**inputs)

# outputs.attentions is a tuple of length num_layers,
# each element shaped (batch=1, num_heads=12, seq_len, seq_len)
attentions = outputs.attentions

print(f"Number of layers returned: {len(attentions)}")
print(f"Shape of one layer's attention tensor: {attentions[0].shape}")
print("  -> (batch_size, num_heads, seq_len, seq_len)")



## Section 5 — Visualize a Single Attention Head as a Heatmap

We'll plot the attention matrix for one layer/head as a heatmap: rows and columns are both the
sentence's tokens, and cell `(i, j)` is colored by how strongly token `i` attends to token `j`.
Bright cells in the "it" row show where the model is pulling context from.

We use a **later layer** (layer 5, the last of 6) because deeper layers in BERT-style models
tend to capture more semantic/coreference-style relationships, while earlier layers often
capture more local syntactic patterns (adjacent words, punctuation).


In [ ]:

def plot_attention_heatmap(attn_matrix, tokens, title):
    # Plot a single (seq_len, seq_len) attention matrix as an annotated heatmap.
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(attn_matrix, cmap="viridis")

    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_xlabel("Attending TO (Key)")
    ax.set_ylabel("Attending FROM (Query)")
    ax.set_title(title)

    fig.colorbar(im, ax=ax, label="Attention weight")
    plt.tight_layout()
    plt.show()

LAYER = 5   # last layer (0-indexed, DistilBERT has 6 layers total)
HEAD = 0    # first head in that layer — we'll compare more heads in Section 7

layer_attn = attentions[LAYER][0]      # drop the batch dimension -> (num_heads, seq_len, seq_len)
single_head_attn = layer_attn[HEAD].numpy()

plot_attention_heatmap(
    single_head_attn,
    tokens,
    title=f"DistilBERT Attention — Layer {LAYER}, Head {HEAD}",
)



## Section 6 — Where Does "it" Actually Attend?

Rather than eyeballing the full heatmap, let's isolate the single row for the token **"it"**
and rank every other token by attention weight. This directly answers our original question:
does the model resolve "it" toward the claim, the Lessee, or the Lessor?

We average across **all 12 heads in the final layer** first — a common technique when you want
a single "consensus" view rather than one head's idiosyncratic pattern.


In [ ]:

# Find the token index for "it" in our tokenized sentence
it_index = tokens.index("it")
print(f"'it' is token index {it_index}: {tokens[it_index]}")

# Average attention across all heads in the last layer -> shape (seq_len, seq_len)
last_layer_attn = attentions[LAYER][0]                  # (num_heads, seq_len, seq_len)
avg_attn_across_heads = last_layer_attn.mean(dim=0)      # (seq_len, seq_len)

# Pull out just the row for "it" -- how much "it" attends to every other token
it_attention_row = avg_attn_across_heads[it_index].detach().numpy()

# Rank tokens by attention weight, highest first
ranked_indices = np.argsort(-it_attention_row)

print("\nTop tokens that 'it' attends to (averaged across heads, last layer):\n")
for rank, idx in enumerate(ranked_indices[:8], start=1):
    print(f"  {rank}. {tokens[idx]:<15s}  weight = {it_attention_row[idx]:.4f}")



**Reading the result:** if the model is doing its job, "it" should assign meaningful attention
weight to tokens like `claim`, `##ing` (from "arising"), or `there`/`##from` — pulling context
from "any claim arising therefrom" — rather than defaulting only to whichever party noun
appears closest. This is the same phenomenon as the classic "the animal didn't cross the
street because it was too tired" example from the deck, just in contract language.

*Note: subword tokens are prefixed with `##` when they continue a previous token (WordPiece
convention) — e.g. "therefrom" may tokenize as `there` + `##from`.*



## Section 7 — Comparing Multiple Heads (Multihead Attention, Made Visible)

The Transformers 101 deck's core claim about **multihead attention** is that different heads
learn to specialize in different relationship types — one head might track syntax (subject →
verb), another might track long-range coreference. Let's visualize 4 different heads from the
same layer side by side and see if their patterns actually differ.


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
heads_to_compare = [0, 3, 6, 9]  # sample 4 of the 12 heads in this layer

for ax, head_idx in zip(axes.flat, heads_to_compare):
    attn_matrix = attentions[LAYER][0][head_idx].detach().numpy()
    im = ax.imshow(attn_matrix, cmap="viridis")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=7)
    ax.set_yticklabels(tokens, fontsize=7)
    ax.set_title(f"Head {head_idx}")

fig.suptitle(f"Layer {LAYER}: Different Heads Attend Differently", fontsize=14)
plt.tight_layout()
plt.show()

print("Compare the 4 panels above: do some heads look more 'diagonal' (attending mostly to")
print("nearby words -- local/syntactic) while others show longer-range bright cells")
print("(long-distance/coreference-style)? That difference IS multihead attention working.")



## Section 8 — Masked Multihead Attention (Causal Masking)

DistilBERT's self-attention above is **bidirectional** — every token can see every other
token. But decoder/generation models (like DistilGPT2, used in the Finetuning notebook) must
be **causally masked**: when predicting token `i`, the model is only allowed to attend to
tokens `1...i`, never anything after it. Otherwise the model could "cheat" by peeking at the
answer it's supposed to generate.

Rather than pull this apart from inside a real model, we build a small standalone example so
the masking mechanism itself is fully visible: take a toy 4-token attention score matrix, apply
a causal mask, and watch what happens at the softmax step.


In [ ]:

# A toy 4-token sequence, e.g. representing: "Party", "shall", "indemnify", "Client"
toy_tokens = ["Party", "shall", "indemnify", "Client"]
seq_len = len(toy_tokens)

# A toy raw attention SCORE matrix (before softmax) -- just illustrative numbers,
# not derived from a real model, so the masking effect is easy to see clearly.
np.random.seed(0)
raw_scores = np.random.uniform(low=0.5, high=2.0, size=(seq_len, seq_len))

print("Raw attention scores (before masking, before softmax):")
print(np.round(raw_scores, 2))


In [ ]:

# Build the causal mask: allowed positions (j <= i) get 0, forbidden future positions
# (j > i) get -infinity, so they vanish after softmax.
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -np.inf
causal_mask = np.nan_to_num(causal_mask, neginf=-1e9)  # numerically-safe stand-in for -inf

print("Causal mask (0 = allowed, large negative = forbidden/future):")
print(causal_mask)

masked_scores = raw_scores + causal_mask
print("\nMasked scores (raw scores + mask):")
print(np.round(masked_scores, 2))


In [ ]:

def softmax(x, axis=-1):
    # Numerically stable softmax: subtract the row max before exponentiating
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

attention_weights = softmax(masked_scores, axis=-1)

print("Final attention weights after softmax (rows sum to 1):")
print(np.round(attention_weights, 3))

print("\nNotice: every row's upper-right (future) positions are exactly 0.0 --")
print("token 'Party' (row 0) attends ONLY to itself; token 'Client' (row 3, the last token)")
print("is the only one allowed to attend across the full sequence.")

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(attention_weights, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(seq_len)); ax.set_xticklabels(toy_tokens, rotation=45)
ax.set_yticks(range(seq_len)); ax.set_yticklabels(toy_tokens)
ax.set_title("Causally-Masked Attention Weights")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()



## Section 9 — Positional Encoding, Made Visible

Self-attention on its own has **no sense of word order** — swapping two tokens' positions
doesn't change how much they attend to each other unless order is injected explicitly. The
original transformer does this with fixed **sinusoidal positional encodings**, added directly
to token embeddings:

```
PE(pos, 2i)   = sin( pos / 10000^(2i / d_model) )
PE(pos, 2i+1) = cos( pos / 10000^(2i / d_model) )
```

Let's compute and plot this pattern directly so the "wave-interference fingerprint per
position" claim from the deck becomes something you can actually see.


In [ ]:

def positional_encoding(seq_len, d_model):
    # position: column vector of shape (seq_len, 1) -- one row per token position
    position = np.arange(seq_len)[:, np.newaxis]
    # dimension index i, used in the 10000^(2i/d_model) denominator term
    dim_index = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1.0 / np.power(10000, (2 * (dim_index // 2)) / np.float32(d_model))
    angle_rads = position * angle_rates

    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle_rads[:, 0::2])   # even dimensions -> sine
    pe[:, 1::2] = np.cos(angle_rads[:, 1::2])   # odd dimensions -> cosine
    return pe

SEQ_LEN = 50    # e.g. up to 50 tokens of a contract clause
D_MODEL = 64    # embedding dimension (kept small here for a readable plot)

pe_matrix = positional_encoding(SEQ_LEN, D_MODEL)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pe_matrix.T, cmap="RdBu", aspect="auto")
ax.set_xlabel("Position in sequence")
ax.set_ylabel("Embedding dimension")
ax.set_title("Sinusoidal Positional Encoding — Wave-Interference Pattern")
fig.colorbar(im, ax=ax, label="Encoding value")
plt.tight_layout()
plt.show()

print("Each row is a different dimension oscillating at a different frequency -- low")
print("dimensions change slowly across positions, high dimensions oscillate quickly.")
print("The COMBINATION across all dimensions gives every position a unique 'fingerprint',")
print("and because it's smooth/periodic, the model can generalize to positions it never")
print("saw during training.")



## Section 10 — Try It Yourself

Swap in your own clause below and re-run Sections 4–6. Good candidates to try:

- A **termination clause** with a dangling "such" or "the foregoing"
- A **governing law clause** with a cross-reference to "the jurisdiction set forth above"
- A clause from a real contract your firm has on file (redact anything confidential first!)


In [ ]:

# Try your own sentence here, then re-run the cells in Sections 4-7 with this variable
your_sentence = (
    "Upon termination of this Agreement, the Contractor shall return all Confidential "
    "Information to the Company, unless it is required to be retained by law."
)

your_inputs = tokenizer(your_sentence, return_tensors="pt")
your_tokens = tokenizer.convert_ids_to_tokens(your_inputs["input_ids"][0])

with torch.no_grad():
    your_outputs = model(**your_inputs)

your_attn = your_outputs.attentions[LAYER][0].mean(dim=0).detach().numpy()
plot_attention_heatmap(your_attn, your_tokens, title="Your Sentence — Averaged Attention, Last Layer")



## Key Takeaways

1. **Self-attention is computable and visible** — it's not a black box; every attention weight
   is just a number from `softmax(QKᵀ/√d_k)`, and we just plotted a whole matrix of them.
2. **Context resolution happens through attention weights**, not hard-coded grammar rules —
   the model "decided" what "it" refers to purely by learning to weight relevant tokens highly
   during pretraining.
3. **Multihead attention gives the model several "lenses"** on the same sentence at once —
   some heads track nearby words, others track long-range relationships.
4. **Causal masking** (Section 8) is what turns bidirectional self-attention into the
   autoregressive, left-to-right generation used by GPT-style models — enforced by adding
   `-infinity` to future positions before softmax.
5. **Positional encoding** (Section 9) is what lets attention know word order at all — without
   it, "the Lessee shall indemnify the Lessor" and "the Lessor shall indemnify the Lessee"
   would look identical to the attention mechanism.
6. This is the exact same mechanism, scaled up, that powers every LLM used later in this
   training (masked/causal versions for generation, cross-attention for translation, etc.).

**Next up:** the *Tokenization* notebook — how "therefrom," "estoppel," and other legal jargon
actually get split into subword tokens before they ever reach this attention mechanism.
